# 🍊 FRESH - YOLOv11s Fruit Detection Training (Production v2)

**Model:** YOLOv11s (Small) - Optimized for accuracy and production deployment  
**Dataset:** fruit-detection.v2i.yolov11 (3,950 images)  
**Classes:** Grapefruit, Guava, Mango, Orange  
**Training Date:** March 7, 2026

---

## 🎯 Training Objectives
- **Replace** underperforming previous model with high-accuracy YOLOv11s
- **Address** class imbalance (2.57:1 ratio) using augmentation
- **Achieve** ≥85% mAP@0.5 and ≥60% mAP@0.5:0.95
- **Ensure** balanced performance across all 4 fruit classes
- **Deliver** production-ready model for FRESH ML pipeline

---

## 📋 Pre-Training Checklist
- [ ] Dataset uploaded to Kaggle
- [ ] Dataset added as input to this notebook
- [ ] GPU accelerator enabled (Settings → Accelerator → GPU)
- [ ] Internet enabled (Settings → Internet → ON)
- [ ] Session persistence set (Settings → Persistence → "Always save output")

---

## 🚀 Let's begin!

## 1️⃣ Import Required Libraries

In [ ]:
# Install Ultralytics YOLOv11 with compatible NumPy
# Force clean install to avoid binary incompatibility
!pip uninstall -y numpy
!pip install --no-cache-dir ultralytics==8.3.0

print("✅ Dependencies installed successfully!")
print("\n⚠️ IMPORTANT: Restart runtime now!")
print("   Kernel → Restart Session")
print("   Then re-run all cells from the top")

In [ ]:
# Import Required Libraries
import os
import yaml
import torch
import shutil
import numpy as np
from pathlib import Path
from collections import Counter
from ultralytics import YOLO
import matplotlib.pyplot as plt

print("✅ Libraries imported successfully!")
print(f"\n📊 Environment Information:")
print(f"  PyTorch Version: {torch.__version__}")
print(f"  CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("  ⚠️ WARNING: No GPU detected! Training will be very slow.")
    print("  Please enable GPU: Settings → Accelerator → GPU")

## 2️⃣ Configure Dataset Paths and Hyperparameters

In [ ]:
# Discover Dataset in Kaggle Input
print("🔍 Searching for dataset in Kaggle input directory...")

# List all available datasets
if os.path.exists('/kaggle/input'):
    datasets = os.listdir('/kaggle/input')
    print(f"\n📁 Available Datasets: {len(datasets)}")
    for ds in datasets:
        print(f"  - {ds}")
    
    # Try to find the fruit detection dataset
    dataset_candidates = [ds for ds in datasets if 'fruit' in ds.lower() or 'detection' in ds.lower()]
    
    if dataset_candidates:
        DATASET_NAME = dataset_candidates[0]
        print(f"\n✅ Auto-detected dataset: {DATASET_NAME}")
    else:
        # Fallback: use first dataset
        DATASET_NAME = datasets[0] if datasets else "unknown"
        print(f"\n⚠️ No fruit dataset found, using: {DATASET_NAME}")
else:
    print("❌ ERROR: /kaggle/input not found!")
    print("Please add your dataset as input to this notebook:")
    print("  1. Click 'Add Input' on the right sidebar")
    print("  2. Search for your uploaded dataset")
    print("  3. Click 'Add'")
    DATASET_NAME = "CHANGE_ME"  # User must update this

print(f"\n🎯 Using dataset: {DATASET_NAME}")

In [ ]:
# Locate Dataset Structure
dataset_base = Path(f'/kaggle/input/{DATASET_NAME}')

print(f"📂 Searching in: {dataset_base}")

# Try to find the dataset root (could be nested)
possible_roots = [
    dataset_base,
    dataset_base / 'hassangill' / 'fruit-detection' / 'fruit-detection.v2i.yolov11',  # Correct nested path
    dataset_base / 'hassangill' / 'fruit-detection.v2i.yolov11',
    dataset_base / 'hassangill',
    dataset_base / 'fruit-detection.v2i.yolov11',
    dataset_base / 'dataset',
    dataset_base / 'yolo',
]

print(f"\n🔎 Checking possible dataset locations...")
dataset_path = None
for root in possible_roots:
    if (root / 'train' / 'images').exists() and (root / 'data.yaml').exists():
        dataset_path = root
        print(f"✅ Found dataset at: {root}")
        break

if dataset_path is None:
    print("❌ ERROR: Dataset structure not found!")
    print(f"\n📋 Contents of {dataset_base}:")
    if dataset_base.exists():
        for item in dataset_base.iterdir():
            print(f"  - {item.name}")
    print("\n⚠️ Please verify:")
    print("  1. Dataset uploaded correctly")
    print("  2. Contains: train/images, train/labels, valid/images, valid/labels, data.yaml")
    raise FileNotFoundError("Dataset not found in expected locations")

print(f"\n✅ Dataset located: {dataset_path}")

In [ ]:
# Verify Dataset Structure & Count Images
required_dirs = [
    'train/images', 'train/labels',
    'valid/images', 'valid/labels',
]

# Note: test set is optional for training
optional_dirs = ['test/images', 'test/labels']

print("🔍 Verifying dataset structure...\n")
print("="*60)

all_required_exist = True
for dir_path in required_dirs:
    full_path = dataset_path / dir_path
    exists = full_path.exists()
    status = "✅" if exists else "❌"
    
    if exists and 'images' in dir_path:
        count = len(list(full_path.glob('*')))
        print(f"{status} {dir_path:20s} - {count:,} files")
    else:
        print(f"{status} {dir_path:20s}")
    
    if not exists:
        all_required_exist = False

# Check optional directories
for dir_path in optional_dirs:
    full_path = dataset_path / dir_path
    if full_path.exists():
        if 'images' in dir_path:
            count = len(list(full_path.glob('*')))
            print(f"✅ {dir_path:20s} - {count:,} files (optional)")

print("="*60)

if not all_required_exist:
    raise RuntimeError("❌ Required directories missing!")
else:
    print("\n✅ Dataset structure verified successfully!")

## 3️⃣ Load and Explore Dataset

In [ ]:
# Analyze Class Distribution
class_names = ['grapefruit', 'guava', 'mango', 'orange']

def analyze_class_distribution(labels_dir, split_name):
    """Analyze class distribution in a dataset split"""
    label_files = list(labels_dir.glob('*.txt'))
    
    if not label_files:
        print(f"⚠️ No labels found in {split_name}")
        return None
    
    # Count classes
    class_counts = Counter()
    total_boxes = 0
    
    for label_file in label_files:
        with open(label_file, 'r') as f:
            for line in f:
                if line.strip():
                    class_id = int(line.split()[0])
                    class_counts[class_id] += 1
                    total_boxes += 1
    
    # Print results
    print(f"\n{split_name.upper()} SET:")
    print(f"  Images: {len(label_files):,}")
    print(f"  Total Boxes: {total_boxes:,}")
    print(f"  Avg Boxes/Image: {total_boxes/len(label_files):.2f}")
    print(f"\n  Class Distribution:")
    
    for class_id in range(4):
        count = class_counts.get(class_id, 0)
        percentage = (count / total_boxes * 100) if total_boxes > 0 else 0
        print(f"    {class_id}: {class_names[class_id]:12s} - {count:5,} boxes ({percentage:5.2f}%)")
    
    return class_counts

print("="*70)
print("📊 DATASET CLASS DISTRIBUTION ANALYSIS")
print("="*70)

# Analyze each split
train_dist = analyze_class_distribution(dataset_path / 'train/labels', 'train')
valid_dist = analyze_class_distribution(dataset_path / 'valid/labels', 'valid')

if (dataset_path / 'test/labels').exists():
    test_dist = analyze_class_distribution(dataset_path / 'test/labels', 'test')

print("\n" + "="*70)

In [ ]:
# Calculate Class Imbalance & Weights
if train_dist:
    print("⚖️ CLASS IMBALANCE ANALYSIS\n")
    print("="*70)
    
    # Get counts
    counts = [train_dist.get(i, 0) for i in range(4)]
    total = sum(counts)
    
    # Calculate imbalance ratio
    max_count = max(counts)
    min_count = min([c for c in counts if c > 0]) if any(counts) else 1
    imbalance_ratio = max_count / min_count
    
    print(f"Most Common Class: {max_count:,} instances ({class_names[counts.index(max_count)]})")
    print(f"Least Common Class: {min_count:,} instances ({class_names[counts.index(min_count)]})")
    print(f"Imbalance Ratio: {imbalance_ratio:.2f}:1")
    
    # Assessment
    if imbalance_ratio < 1.5:
        print("✅ Status: Well-balanced dataset")
    elif imbalance_ratio < 3:
        print("⚠️ Status: Moderate imbalance - augmentation will help")
    else:
        print("❌ Status: Significant imbalance - heavy augmentation required")
    
    # Calculate inverse frequency weights (for reference)
    print(f"\n📐 CALCULATED CLASS WEIGHTS (Inverse Frequency):\n")
    weights = {}
    for i in range(4):
        count = counts[i]
        if count > 0:
            weight = total / (4 * count)
            weights[i] = weight
            print(f"  Class {i} ({class_names[i]:12s}): {weight:.3f}")
        else:
            weights[i] = 1.0
            print(f"  Class {i} ({class_names[i]:12s}): 1.000 (no samples!)")
    
    print("\n💡 These weights are implicit through heavy augmentation (mosaic + mixup)")
    print("   Mosaic will effectively boost underrepresented classes like mango")
    print("="*70)

## 4️⃣ Define Data Augmentation
**Critical for handling class imbalance!**

In [ ]:
# Create data.yaml Configuration
yaml_config = {
    'path': str(dataset_path),
    'train': 'train/images',
    'val': 'valid/images',
    'nc': 4,
    'names': class_names  # ['grapefruit', 'guava', 'mango', 'orange']
}

# Add test if available
if (dataset_path / 'test/images').exists():
    yaml_config['test'] = 'test/images'

# Save to working directory (writable)
yaml_file = Path('/kaggle/working/data.yaml')
with open(yaml_file, 'w') as f:
    yaml.dump(yaml_config, f, default_flow_style=False, sort_keys=False)

print("✅ Created data.yaml configuration\n")
print("="*70)
print("📄 Configuration:")
print("="*70)
with open(yaml_file, 'r') as f:
    print(f.read())
print("="*70)

print(f"\n🎯 Training will use:")
print(f"  Classes: {yaml_config['names']}")
print(f"  Class IDs: 0=grapefruit, 1=guava, 2=mango, 3=orange")
print(f"\n⚠️ IMPORTANT: After training, update pipeline class mapping to match!")

## 5️⃣ Configure Model Architecture

**Decision: YOLOv11s (Small)**

Why YOLOv11s over YOLOv11n?
- **+8% better mAP** (45% vs 37% typical)
- **Better small object detection** (distant fruits in orchard)
- **More robust** to lighting/angle variations
- **Production-ready accuracy** vs just "good enough"
- Still fast: ~40 FPS (vs 50 FPS for nano)

For production quality where accuracy issues led to retraining, the 2-3hr extra training time is worth it.

In [ ]:
# Initialize YOLOv11s Model
print("🤖 Initializing YOLOv11s model...\n")

# Download pretrained COCO weights
model = YOLO('yolo11s.pt')  # YOLOv11 small variant

print("✅ Model initialized successfully!")
print(f"\nℹ️ Model Information:")
print(f"  Architecture: YOLOv11s (Small)")
print(f"  Parameters: ~9.4M")
print(f"  Pretrained: COCO dataset (80 classes)")
print(f"  Will be fine-tuned for 4 fruit classes")
print(f"\n🎯 Why YOLOv11s?")
print(f"  ✅ Better accuracy than nano variant (+8% mAP)")
print(f"  ✅ Still fast enough for production (~40 FPS)")
print(f"  ✅ Better small object detection (distant fruits)")
print(f"  ✅ More robust to variations in lighting/angle")

## 6️⃣ Setup Training with Early Stopping

In [ ]:
# Training Hyperparameters
# Optimized for class-imbalanced fruit detection

training_config = {
    # Model & Data
    'data': str(yaml_file),
    'epochs': 150,
    'batch': 16,  # Reduce to 8 if GPU memory issues
    'imgsz': 512,  # Matches dataset preprocessing
    'device': 0,  # GPU (use 'cpu' if no GPU)
    
    # Optimizer
    'optimizer': 'AdamW',  # Better than SGD for this task
    'lr0': 0.001,  # Initial learning rate
    'lrf': 0.01,   # Final learning rate (1% of initial)
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3.0,
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    
    # Loss weights
    'box': 7.5,    # Box loss gain
    'cls': 0.5,    # Class loss gain
    'dfl': 1.5,    # DFL loss gain
    
    # Augmentation (CRITICAL for imbalance)
    'hsv_h': 0.015,      # Hue augmentation (ripeness variation)
    'hsv_s': 0.7,        # Saturation augmentation
    'hsv_v': 0.4,        # Value (brightness) augmentation
    'degrees': 10.0,     # Rotation (+/- degrees)
    'translate': 0.1,    # Translation (+/- fraction)
    'scale': 0.5,        # Scaling (+/- gain)
    'shear': 2.0,        # Shear (+/- degrees)
    'perspective': 0.0001,  # Perspective transformation
    'flipud': 0.0,       # Vertical flip (NO - fruits don't grow upside down)
    'fliplr': 0.5,       # Horizontal flip (50% probability)
    'mosaic': 1.0,       # Mosaic augmentation (mix 4 images - CRITICAL for mango)
    'mixup': 0.15,       # Mixup augmentation (blend images)
    'copy_paste': 0.1,   # Copy-paste augmentation
    
    # Training settings
    'patience': 25,      # Early stopping patience
    'save': True,        # Save checkpoints
    'save_period': -1,   # Save every N epochs (-1 = disabled, only best)
    'cache': False,      # Don't cache images (Kaggle memory limit)
    'workers': 8,        # DataLoader workers
    'project': 'FRESH_FruitDetection',  # WandB-friendly project name
    'name': 'yolo11s_training',
    'exist_ok': True,
    'pretrained': True,
    'verbose': True,
    'seed': 42,
    'deterministic': False,  # Faster training
    'single_cls': False,
    'rect': False,       # Rectangular training (disabled for augmentation)
    'cos_lr': True,      # Cosine learning rate scheduler
    'close_mosaic': 10,  # Disable mosaic in last N epochs for stability
    'amp': True,         # Automatic Mixed Precision (faster training)
    'fraction': 1.0,     # Use 100% of dataset
    'profile': False,
    'freeze': None,      # Don't freeze layers
    'multi_scale': False,  # Disabled for consistent 512x512
    'overlap_mask': True,
    'mask_ratio': 4,
    'dropout': 0.0,
    'val': True,         # Run validation
    'plots': True,       # Generate plots
}

print("⚙️ TRAINING CONFIGURATION")
print("="*70)
print("\n📊 Key Settings:")
print(f"  Model: YOLOv11s")
print(f"  Epochs: {training_config['epochs']}")
print(f"  Batch Size: {training_config['batch']}")
print(f"  Image Size: {training_config['imgsz']}x{training_config['imgsz']}")
print(f"  Optimizer: {training_config['optimizer']}")
print(f"  Initial LR: {training_config['lr0']}")
print(f"  Early Stopping: {training_config['patience']} epochs")
print(f"\n🎨 Augmentation Strategy:")
print(f"  Mosaic: {training_config['mosaic']} (mix 4 images - boost mango class)")
print(f"  Mixup: {training_config['mixup']} (blend images)")
print(f"  Flip Horizontal: {training_config['fliplr']*100:.0f}%")
print(f"  Rotation: ±{training_config['degrees']}°")
print(f"  Color Variation: H={training_config['hsv_h']}, S={training_config['hsv_s']}, V={training_config['hsv_v']}")
print(f"\n🎯 Performance Targets:")
print(f"  mAP@0.5: ≥85%")
print(f"  mAP@0.5:0.95: ≥60%")
print(f"  All Classes Recall: ≥70%")
print("="*70)

## 7️⃣ Train the Model
**⚠️ This will take approximately 4-6 hours on Kaggle GPU (P100/T4)**

The training process includes:
- Automatic validation every epoch
- Best model saved based on mAP@0.5:0.95
- Early stopping after 25 epochs without improvement
- Comprehensive metrics logging

**Do NOT close this tab - training will stop!**

In [ ]:
# Start Training
print("🚀 Starting YOLOv11s Training...")
print("="*70)
print("⏱️ Estimated Time: 4-6 hours")
print("📊 Monitor the progress below")
print("⚠️ Do NOT close this tab - training will stop!")
print("="*70)
print("\n")

# Disable Ray Tune callbacks (Kaggle compatibility fix v2)
# Ray's internal API changed - patch it to prevent AttributeError
try:
    import sys
    import ray.train._internal.session
    
    # Create the missing _get_session function that returns None
    if not hasattr(ray.train._internal.session, '_get_session'):
        ray.train._internal.session._get_session = lambda: None
    
    print("✅ Ray Tune compatibility patch applied")
except ImportError:
    # Ray not installed - no patching needed
    pass
except Exception as e:
    print(f"⚠️ Ray Tune patch warning (safe to ignore): {e}")

# Train the model
results = model.train(**training_config)

print("\n")
print("="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)

## 8️⃣ Evaluate Model Performance

In [ ]:
# Load Training Results
results_dir = Path('/kaggle/working/FRESH_FruitDetection/yolo11s_training')
results_csv = results_dir / 'results.csv'

if results_csv.exists():
    import pandas as pd
    
    # Read results
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()  # Remove whitespace
    
    print("📊 TRAINING RESULTS SUMMARY")
    print("="*70)
    
    # Get final metrics
    final_metrics = df.iloc[-1]
    
    print(f"\n🎯 Final Metrics (Last Epoch):")
    if 'metrics/mAP50(B)' in df.columns:
        print(f"  mAP@0.5:        {final_metrics['metrics/mAP50(B)']:.4f}")
    if 'metrics/mAP50-95(B)' in df.columns:
        print(f"  mAP@0.5:0.95:   {final_metrics['metrics/mAP50-95(B)']:.4f}")
    if 'metrics/precision(B)' in df.columns:
        print(f"  Precision:      {final_metrics['metrics/precision(B)']:.4f}")
    if 'metrics/recall(B)' in df.columns:
        print(f"  Recall:         {final_metrics['metrics/recall(B)']:.4f}")
    
    # Get best metrics
    print(f"\n⭐ Best Metrics:")
    if 'metrics/mAP50(B)' in df.columns:
        best_map50 = df['metrics/mAP50(B)'].max()
        print(f"  Best mAP@0.5:        {best_map50:.4f}")
    if 'metrics/mAP50-95(B)' in df.columns:
        best_map50_95 = df['metrics/mAP50-95(B)'].max()
        best_epoch = df['metrics/mAP50-95(B)'].idxmax()
        print(f"  Best mAP@0.5:0.95:   {best_map50_95:.4f} (epoch {best_epoch})")
    
    # Check if targets met
    print(f"\n✅ Target Achievement:")
    if 'metrics/mAP50(B)' in df.columns and best_map50 >= 0.85:
        print(f"  ✅ mAP@0.5 ≥ 85%: ACHIEVED ({best_map50:.1%})")
    elif 'metrics/mAP50(B)' in df.columns:
        print(f"  ⚠️ mAP@0.5 ≥ 85%: {best_map50:.1%} (target: 85%)")
    
    if 'metrics/mAP50-95(B)' in df.columns and best_map50_95 >= 0.60:
        print(f"  ✅ mAP@0.5:0.95 ≥ 60%: ACHIEVED ({best_map50_95:.1%})")
    elif 'metrics/mAP50-95(B)' in df.columns:
        print(f"  ⚠️ mAP@0.5:0.95 ≥ 60%: {best_map50_95:.1%} (target: 60%)")
    
    print("="*70)
else:
    print("⚠️ Results CSV not found - check training logs above")

In [ ]:
# Display Training Curves
from IPython.display import Image, display

print("📈 TRAINING VISUALIZATION")
print("="*70)

# Results plot (loss curves, mAP, etc.)
results_plot = results_dir / 'results.png'
if results_plot.exists():
    print("\n📊 Training Metrics Over Time:")
    display(Image(filename=str(results_plot)))
else:
    print("⚠️ Results plot not found")

# Confusion Matrix
confusion_matrix = results_dir / 'confusion_matrix.png'
if confusion_matrix.exists():
    print("\n🔍 Confusion Matrix (Per-Class Performance):")
    display(Image(filename=str(confusion_matrix)))
else:
    print("⚠️ Confusion matrix not found")

# PR Curve
pr_curve = results_dir / 'PR_curve.png'
if pr_curve.exists():
    print("\n📉 Precision-Recall Curve:")
    display(Image(filename=str(pr_curve)))
else:
    print("⚠️ PR curve not found")

print("="*70)

## 9️⃣ Export Model for Inference

In [ ]:
# Locate Best Model
best_model_path = results_dir / 'weights' / 'best.pt'
last_model_path = results_dir / 'weights' / 'last.pt'

print("💾 MODEL EXPORT")
print("="*70)

if best_model_path.exists():
    # Get model file size
    size_mb = best_model_path.stat().st_size / (1024 * 1024)
    
    print(f"\n✅ Best Model Located:")
    print(f"  Path: {best_model_path}")
    print(f"  Size: {size_mb:.2f} MB")
    
    # Copy to easy-to-find location
    export_path = Path('/kaggle/working/yolo_detection_best_v2.pt')
    shutil.copy(best_model_path, export_path)
    
    print(f"\n📦 Exported Model:")
    print(f"  Path: {export_path}")
    print(f"  Download this file from Kaggle output!")
    
    print(f"\n📋 Model Specifications:")
    print(f"  Architecture: YOLOv11s")
    print(f"  Classes: {class_names}")
    print(f"  Class IDs: 0=grapefruit, 1=guava, 2=mango, 3=orange")
    print(f"  Input Size: 512x512 RGB")
    print(f"  Format: PyTorch (.pt)")
    
else:
    print("❌ ERROR: Best model not found!")
    print(f"  Expected at: {best_model_path}")
    
    if last_model_path.exists():
        print(f"\n⚠️ Last checkpoint found: {last_model_path}")
        print(f"  You can use this if best model is missing")

print("="*70)

## 🔟 Visualize Predictions
Test the trained model on sample images from the validation set

In [ ]:
# Test Inference on Sample Image
if best_model_path.exists():
    print("🧪 INFERENCE TEST")
    print("="*70)
    
    # Load best model
    trained_model = YOLO(str(best_model_path))
    
    # Get a random validation image
    valid_images_dir = dataset_path / 'valid/images'
    sample_images = list(valid_images_dir.glob('*.jpg'))[:3]
    
    if sample_images:
        print(f"\n📸 Testing on {len(sample_images)} sample images...\n")
        
        for i, img_path in enumerate(sample_images, 1):
            print(f"\nImage {i}: {img_path.name}")
            
            # Run inference
            results = trained_model(str(img_path), verbose=False)
            
            # Print detections
            detections = results[0].boxes
            if len(detections) > 0:
                print(f"  Found {len(detections)} fruits:")
                for det in detections:
                    cls_id = int(det.cls)
                    conf = float(det.conf)
                    print(f"    - {class_names[cls_id]:12s} (confidence: {conf:.3f})")
            else:
                print(f"  ⚠️ No fruits detected")
        
        # Save annotated image
        results = trained_model(str(sample_images[0]))
        annotated = results[0].plot()
        
        # Save to working directory
        test_output = Path('/kaggle/working/test_detection.jpg')
        import cv2
        cv2.imwrite(str(test_output), annotated)
        
        print(f"\n📊 Annotated image saved: {test_output}")
        print(f"   View in Kaggle output to see detection boxes!")
        
    else:
        print("⚠️ No validation images found for testing")
    
    print("="*70)
else:
    print("⚠️ Cannot run inference test - model file not found")

## ✅ Training Complete - Next Steps

### 📥 Download Your Model
1. In Kaggle, click **"Output"** tab (top right)
2. Find and download: `yolo_detection_best_v2.pt`
3. Save to your FRESH_ML project

### 🔄 Update FRESH ML Pipeline

**Step 1:** Update class mapping in `pipeline/detection/yolo_detector.py`:
```python
# Change this:
self.class_names = {
    0: "mango",
    1: "orange", 
    2: "guava",
    3: "grapefruit"
}

# To this (match new model):
self.class_names = {
    0: "grapefruit",
    1: "guava",
    2: "mango",
    3: "orange"
}
```

**Step 2:** Replace model file:
```bash
# Backup old model
mv FRESH_ML/models/yolo_detection_best.pt FRESH_ML/models/yolo_detection_best_old.pt

# Copy new model
cp yolo_detection_best_v2.pt FRESH_ML/models/yolo_detection_best.pt
```

**Step 3:** Test the updated pipeline:
```bash
cd FRESH_ML
python -m pytest tests/test_yolo_detector.py
```

### 🎯 Success Criteria Checklist
- [ ] Model downloaded from Kaggle
- [ ] mAP@0.5 ≥ 85% achieved
- [ ] Pipeline class mapping updated
- [ ] Model file replaced in FRESH_ML/models/
- [ ] Pipeline tests passing
- [ ] Inference working on sample images

---

## 🎉 Congratulations!
You've successfully trained a production-ready YOLOv11s fruit detection model with:
- ✅ Class imbalance handling via augmentation
- ✅ Optimized hyperparameters for your dataset
- ✅ Early stopping to prevent overfitting
- ✅ Comprehensive metrics and visualization

**Now deploy your model and start detecting fruits! 🍊🥭🍈**